# N06 — Rejection cutoff + teacher complexity ablations

**Purpose** (two ablations in one notebook):
1. AdvTop-k stability across reject percentiles {95, 90, 80, 70, 50}.
2. Teacher depth {1, 3, 6, unlimited} impact on surrogate fidelity.

**Outputs** (`outputs/N06/`): `cutoff_results.json`, `complexity_results.json`

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import pickle, json, numpy as np, pandas as pd
from pathlib import Path
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from decentra.surrogate import TreeSurrogate, EBMSurrogate, LinearSurrogate
from decentra.metrics.named import advtopk_named, advfull_named
from decentra._utils import transform_logit_to_score

N01 = Path('../outputs/N01'); N02 = Path('../outputs/N02')
OUT = Path('../outputs/N06'); OUT.mkdir(parents=True, exist_ok=True)
with open(N01/'datasets.pkl','rb') as f: datasets = pickle.load(f)

## 1. Cutoff sensitivity (on already-trained surrogates from N02)

In [ ]:
# Use N03 bench pickle if available; otherwise retrain Tree-d1 quickly
PCTS = [95, 90, 80, 70, 50]
cutoff_rows = []
for name, d in datasets.items():
    shap_te = np.load(N02/f'bb_shap_{name}.npy')
    prob_te = np.load(N02/f'bb_prob_{name}.npy')
    score_tr = np.load(N02/f'bb_score_train_{name}.npy')
    with open(N02/f'train_val_idx_{name}.pkl','rb') as f: tv = pickle.load(f)
    feats = d['feature_names']; bb_adv = pd.DataFrame(shap_te, columns=feats)

    for sname, factory in [
        ('Tree-d1',  lambda: TreeSurrogate(max_depth=1, verbose=0)),
        ('EBM',      lambda: EBMSurrogate(interactions=0, n_jobs=8)),
        ('Ridge',    lambda: LinearSurrogate(method='ridge', alpha=1.0)),
    ]:
        surr = factory()
        X_tr_i = d['X_train'].iloc[tv['train_pos']]; X_val = d['X_train'].iloc[tv['val_pos']]
        y_tr = score_tr[tv['train_pos']]; y_val = score_tr[tv['val_pos']]
        try: surr.fit(X_tr_i, y_tr, eval_set=(X_val, y_val))
        except TypeError: surr.fit(X_tr_i, y_tr)
        adv = surr.adverse_contributions(d['X_test'], target_scale='score')
        for pct in PCTS:
            reject = prob_te >= np.percentile(prob_te, pct)
            row = {'dataset': name, 'surrogate': sname, 'pct': pct, 'n_reject': int(reject.sum())}
            row['AdvTop1'] = advtopk_named(bb_adv, adv, reject, 1)
            row['AdvTop4'] = advtopk_named(bb_adv, adv, reject, 4)
            r, j = advfull_named(bb_adv, adv, reject)
            row['AdvFull_R'] = r; row['AdvFull_J'] = j
            cutoff_rows.append(row)

pd.DataFrame(cutoff_rows).to_json(OUT/'cutoff_results.json', orient='records', indent=2)
print(pd.DataFrame(cutoff_rows).round(4).to_string(index=False))

## 2. Teacher complexity (depth 1 / 3 / 6 / unlimited)

In [ ]:
from decentra.experiments import BenchmarkConfig, run_benchmark
import shap

DEPTHS = [1, 3, 6, -1]
cx_rows = []
for name, d in datasets.items():
    for depth in DEPTHS:
        clf = lgb.LGBMClassifier(max_depth=depth, num_leaves=63 if depth==-1 else 2**depth,
                                  n_estimators=300, learning_rate=0.05, subsample=0.8,
                                  colsample_bytree=0.8, min_child_samples=50,
                                  random_state=42, n_jobs=-1)
        X_tr_i, X_val, y_tr_i, y_val = train_test_split(
            d['X_train'], d['y_train'], test_size=0.2,
            stratify=d['y_train'], random_state=42)
        clf.fit(X_tr_i, y_tr_i, eval_set=[(X_val, y_val)],
                callbacks=[lgb.early_stopping(50, verbose=False)])
        prob_te = clf.predict_proba(d['X_test'])[:, 1]
        shap_vals = shap.TreeExplainer(clf).shap_values(d['X_test'])
        if isinstance(shap_vals, list): shap_vals = shap_vals[1]
        score_te = transform_logit_to_score(prob_te)
        score_tr = transform_logit_to_score(clf.predict_proba(d['X_train'])[:, 1])

        cfg = BenchmarkConfig(
            surrogates={'Tree-d1': lambda fn: TreeSurrogate(max_depth=1, verbose=0)},
            reject_percentile=90, target_scale='score',
        )
        res = run_benchmark(
            teacher=None,
            X_train=d['X_train'], X_test=d['X_test'],
            y_train_target=score_tr, y_test_binary=d['y_test'].values,
            bb_shap_test=shap_vals, bb_prob_test=prob_te, bb_score_test=score_te,
            feature_names=d['feature_names'], config=cfg,
        )
        r = res.rows[0]
        cx_rows.append({'dataset': name, 'teacher_depth': depth,
                         'R2': r['R2'], 'AdvTop1': r['AdvTop1'], 'AdvTop4': r['AdvTop4']})
        print(f'{name} depth={depth}: R²={r["R2"]:.4f} AT1={r["AdvTop1"]:.3f} AT4={r["AdvTop4"]:.3f}')

pd.DataFrame(cx_rows).to_json(OUT/'complexity_results.json', orient='records', indent=2)